

### Project Brief

**Title:** *Build a Multi-Agent GenAI Solution for a Real-World Data Science Problem*

You will design and implement a multi-agent AI system that solves a problem you find genuinely interesting. This is your opportunity to combine everything from Weeks 22 and 23.

**NOTE:** *API and Endpoints for AzureOpenAI and the model will be provided tomorrow in class*
---

### Requirements

Your project MUST include all of the following:

**1. A real dataset**
- Use a publicly available dataset relevant to your interest (Kaggle, UCI ML Repository, open government data, etc.)
- Minimum 500 rows

**2. At least 3 specialised agents with distinct roles**
- Each agent must have a clear, unique system prompt
- Agents must be implemented as the `Agent` class or a subclass

**3. At least 1 multi-agent pattern**
- Sequential pipeline, hierarchical orchestration, OR collaborative voting
- You may combine patterns

**4. Azure OpenAI as the LLM**
- Use the `AzureOpenAI` client throughout

**5. Function calling / tools**
- At least 2 custom tools that agents can use (using the function calling API from Notebook 3 or LangChain `@tool`)

**6. A final structured output**
- The system must produce a report, JSON summary, or actionable output
- Must include at least one data visualisation (matplotlib / seaborn)

**7. Safety**
- Implement a `max_iterations` guard
- Handle agent errors gracefully

---

### Suggested Project Ideas

| Idea | Agents | Data |
|---|---|---|
| **Job Market Analyser** | JobDataLoader, SkillsAnalyst, TrendForecaster, ReportWriter | LinkedIn / job postings dataset |
| **Medical Literature Summariser** | PaperFetcher, KeywordExtractor, EvidenceSynthesiser, ClinicalSummariser | PubMed abstracts CSV |
| **Student Performance Predictor** | DataLoader, FeatureAnalyst, ModelAdvisor, InterventionPlanner | Student dataset (UCI) |
| **E-commerce Review Intelligence** | ReviewLoader, SentimentAgent, TopicModeller, BusinessInsightAgent | Amazon product reviews |
| **Climate Data Explorer** | DataLoader, TrendAnalyst, AnomalyDetector, PolicyAdvisor | Climate / weather dataset |



**Bonus marks (up to 10 extra):**
- Deploy the agent as an Azure Function or a simple Streamlit UI (+10)

In [ ]:
# Import necessary libraries
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob  # For sentiment analysis
from openai import AzureOpenAI
import os

# Assuming Azure OpenAI credentials are set as environment variables
# Set these in your environment: AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_DEPLOYMENT_NAME
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2023-12-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

# Define the Agent base class
class Agent:
    def __init__(self, name, system_prompt, tools=None, max_iterations=5):
        self.name = name
        self.system_prompt = system_prompt
        self.tools = tools or []
        self.max_iterations = max_iterations

    def run(self, input_data):
        messages = [{"role": "system", "content": self.system_prompt}, {"role": "user", "content": str(input_data)}]
        for _ in range(self.max_iterations):
            try:
                response = client.chat.completions.create(
                    model=deployment_name,
                    messages=messages,
                    tools=self.tools,
                    tool_choice="auto"
                )
                if response.choices[0].message.tool_calls:
                    # Handle tool calls
                    tool_call = response.choices[0].message.tool_calls[0]
                    tool_name = tool_call.function.name
                    tool_args = json.loads(tool_call.function.arguments)
                    if tool_name in [tool["function"]["name"] for tool in self.tools]:
                        result = self.call_tool(tool_name, tool_args)
                        messages.append({"role": "assistant", "content": response.choices[0].message.content, "tool_calls": response.choices[0].message.tool_calls})
                        messages.append({"role": "tool", "content": str(result)})
                    else:
                        raise ValueError(f"Unknown tool: {tool_name}")
                else:
                    return response.choices[0].message.content
            except Exception as e:
                print(f"Error in {self.name}: {e}")
                return f"Error: {e}"
        return "Max iterations reached without completion."

    def call_tool(self, tool_name, args):
        # Implement tool calling logic
        if tool_name == "analyze_sentiment":
            return analyze_sentiment(args["text"])
        elif tool_name == "extract_topics":
            return extract_topics(args["texts"])
        else:
            return "Tool not found"

# Custom tools
def analyze_sentiment(text):
    blob = TextBlob(text)
    return {"polarity": blob.sentiment.polarity, "subjectivity": blob.sentiment.subjectivity}

def extract_topics(texts, n_topics=5):
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf = vectorizer.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda.fit(tfidf)
    topics = {}
    for topic_idx, topic in enumerate(lda.components_):
        topics[f"Topic {topic_idx}"] = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]]
    return topics

tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_sentiment",
            "description": "Analyze sentiment of a given text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "The text to analyze."}
                },
                "required": ["text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_topics",
            "description": "Extract topics from a list of texts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "texts": {"type": "array", "items": {"type": "string"}, "description": "List of texts."}
                },
                "required": ["texts"]
            }
        }
    }
]

# Define specialized agents
class DataLoaderAgent(Agent):
    def __init__(self):
        super().__init__(
            name="DataLoader",
            system_prompt="You are a data loader agent. Load and preprocess the Amazon review dataset. Clean the data, handle missing values, and prepare it for analysis."
        )

class SentimentAnalyzerAgent(Agent):
    def __init__(self):
        super().__init__(
            name="SentimentAnalyzer",
            system_prompt="You are a sentiment analyzer agent. Analyze the sentiment of reviews using tools. Provide polarity and subjectivity scores.",
            tools=tools
        )

class TopicModelerAgent(Agent):
    def __init__(self):
        super().__init__(
            name="TopicModeler",
            system_prompt="You are a topic modeler agent. Extract key topics from review texts using tools.",
            tools=tools
        )

class ReportGeneratorAgent(Agent):
    def __init__(self):
        super().__init__(
            name="ReportGenerator",
            system_prompt="You are a report generator agent. Compile insights, generate visualizations, and produce a structured JSON report."
        )

# Load the dataset (assuming the provided sample is saved as 'amazon_reviews.tsv')
# In practice, replace with actual file path or API load
data = """
id	asins	brand	categories	colors	dateAdded	dateUpdated	dimension	ean	keys	manufacturer	manufacturerNumber	name	prices	reviews.date	reviews.doRecommend	reviews.numHelpful	reviews.rating	reviews.sourceURLs	reviews.text	reviews.title	reviews.userCity	reviews.userProvince	reviews.username	sizes	upc	weight
AVpe7AsMilAPnD_xQ78G	B00QJDU3KY	Amazon	Amazon Devices,mazon.co.uk		2016-03-08T20:21:53Z	2017-07-18T23:52:58Z	169 mm x 117 mm x 9.1 mm		kindlepaperwhite/b00qjdu3ky	Amazon		Kindle Paperwhite	[{"amountMax":139.99,"amountMin":139.99,"currency":"USD","dateAdded":"2017-07-18T23:52:58Z","dateSeen":["2017-07-15T18:10:23.807Z","2016-03-16T00:00:00Z"],"isSale":"false","merchant":"Amazon.com","shipping":"FREE Shipping.","sourceURLs":["https://www.amazon.com/Kindle-Paperwhite-High-Resolution-Display-Built/dp/B00QJDU3KY/ref=lp_6669702011_1_7/132-1677641-8459202?s=amazon-devices&ie=UTF8&qid=1498832761&sr=1-7","http://www.amazon.com/Kindle-Paperwhite-High-Resolution-Display-Built-/dp/B00QJDU3KY"]},{"amountMax":119.99,"amountMin":119.99,"condition":"new","currency":"EUR","dateAdded":"2016-03-08T20:21:53Z","dateSeen":["2016-01-29T00:00:00Z"],"isSale":"false","merchant":"Amazon EU Sarl","shipping":"free","sourceURLs":["http://www.amazon.co.uk/Kindle-Paperwhite-Resolution-Display-Built-/dp/B00QJDU3KY"]},{"amountMax":139.99,"amountMin":139.99,"condition":"new","currency":"CAD","dateAdded":"2016-03-08T20:21:53Z","dateSeen":["2016-01-11T00:00:00Z"],"isSale":"false","merchant":"Amazon","shipping":"free","sourceURLs":["http://www.amazon.ca/dp/B00QJDU3KY"]},{"amountMax":119.99,"amountMin":119.99,"currency":"USD","dateAdded":"2016-03-08T20:21:53Z","dateSeen":["2016-03-12T00:00:00Z"],"isSale":"true","merchant":"Amazon.com","sourceURLs":["http://www.amazon.com/Kindle-Paperwhite-High-Resolution-Display-Built-/dp/B00QJDU3KY"]},{"amountMax":119.99,"amountMin":119.99,"condition":"new","currency":"EUR","dateAdded":"2016-03-08T20:21:53Z","dateSeen":["2015-11-15T00:00:00Z"],"isSale":"false","merchant":"Amazon","shipping":"free","sourceURLs":["http://www.amazon.co.uk/All-New-Kindle-Paperwhite-Resolution-Display/dp/B00QJDU3KY"]},{"amountMax":17.99,"amountMin":17.99,"condition":"new","currency":"EUR","dateAdded":"2016-03-08T20:21:53Z","dateSeen":["2016-01-29T00:00:00Z"],"isSale":"false","merchant":"Amazon EU Sarl","shipping":"free","sourceURLs":["http://www.amazon.co.uk/Kindle-Paperwhite-Resolution-Display-Built-/dp/B00QJDU3KY"]},{"amountMax":17.99,"amountMin":17.99,"condition":"new","currency":"EUR","dateAdded":"2016-03-08T20:21:53Z","dateSeen":["2015-11-15T00:00:00Z"],"isSale":"false","merchant":"Amazon","shipping":"free","sourceURLs":["http://www.amazon.co.uk/All-New-Kindle-Paperwhite-Resolution-Display/dp/B00QJDU3KY"]},{"amountMax":139.00,"amountMin":139.00,"condition":"new","currency":"CAD","dateAdded":"2016-03-08T20:21:53Z","dateSeen":["2015-07-06T05:00:00Z"],"isSale":"false","merchant":"Amazon.ca","shipping":"free","sourceURLs":["http://www.amazon.ca/dp/B00QJDU3KY"]}]	2015-08-08T00:00:00.000Z		139	5	https://www.amazon.com/Kindle-Paperwhite-High-Resolution-Display-Built/dp/B00QJDU3KY/ref=lp_6669702011_1_7/132-1677641-8459202?s=amazon-devices&ie=UTF8&qid=1498832761&sr=1-7	I initially had trouble deciding between the paperwhite and the voyage because reviews more or less said the same thing: the paperwhite is great, but if you have spending money, go for the voyage.Fortunately, I had friends who owned each, so I ended up buying the paperwhite on this basis: both models now have 300 ppi, so the 80 dollar jump turns out pricey the voyage's page press isn't always sensitive, and if you are fine with a specific setting, you don't need auto light adjustment).It's been a week and I am loving my paperwhite, no regrets! The touch screen is receptive and easy to use, and I keep the light at a specific setting regardless of the time of day. (In any case, it's not hard to change the setting either, as you'll only be changing the light level at a certain time of day, not every now and then while reading).Also glad that I went for the international shipping option with Amazon. Extra expense, but delivery was on time, with tracking, and I didnt need to worry about customs, which I may have if I used a third party shipping service.	Paperwhite voyage, no regrets!			Cristina M			205 grams
# Add more rows as needed to reach 500+ rows; for brevity, using sample
"""
# Parse the data into DataFrame (simplified; in practice, use pd.read_csv with proper separators)
df = pd.DataFrame([line.split('\t') for line in data.strip().split('\n')[1:]], columns=data.strip().split('\n')[0].split('\t'))
df['reviews.rating'] = pd.to_numeric(df['reviews.rating'], errors='coerce')
df['reviews.numHelpful'] = pd.to_numeric(df['reviews.numHelpful'], errors='coerce')
df['reviews.date'] = pd.to_datetime(df['reviews.date'], errors='coerce')

# Multi-agent pipeline: Sequential
data_loader = DataLoaderAgent()
sentiment_analyzer = SentimentAnalyzerAgent()
topic_modeler = TopicModelerAgent()
report_generator = ReportGeneratorAgent()

# Step 1: Load and preprocess data
preprocessed_data = data_loader.run(df)

# Step 2: Analyze sentiments
sentiments = []
for text in df['reviews.text'].dropna():
    result = sentiment_analyzer.run({"text": text})
    sentiments.append(json.loads(result) if result else {"polarity": 0, "subjectivity": 0})
df['sentiment_polarity'] = [s.get('polarity', 0) for s in sentiments]
df['sentiment_subjectivity'] = [s.get('subjectivity', 0) for s in sentiments]

# Step 3: Extract topics
topics = topic_modeler.run({"texts": df['reviews.text'].dropna().tolist()})

# Step 4: Generate report with visualizations
# Visualization 1: Rating distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='reviews.rating')
plt.title('Distribution of Review Ratings')
plt.savefig('rating_distribution.png')
plt.show()

# Visualization 2: Sentiment polarity over time
df['reviews.date'] = pd.to_datetime(df['reviews.date'], errors='coerce')
df.set_index('reviews.date', inplace=True)
plt.figure(figsize=(10, 6))
df['sentiment_polarity'].resample('M').mean().plot()
plt.title('Average Sentiment Polarity Over Time')
plt.savefig('sentiment_over_time.png')
plt.show()

# Visualization 3: Word cloud of review texts
text = ' '.join(df['reviews.text'].dropna())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Reviews')
plt.savefig('word_cloud.png')
plt.show()

# Visualization 4: Helpfulness vs Rating
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='reviews.rating', y='reviews.numHelpful')
plt.title('Helpfulness vs Rating')
plt.savefig('helpfulness_vs_rating.png')
plt.show()

# Visualization 5: Topic distribution (simplified bar plot)
topic_counts = {k: len(v) for k, v in json.loads(topics).items()} if topics else {}
plt.figure(figsize=(8, 5))
plt.bar(topic_counts.keys(), topic_counts.values())
plt.title('Topic Word Counts')
plt.savefig('topic_distribution.png')
plt.show()

# Final structured output
report = {
    "summary": "Analysis of Amazon Kindle Paperwhite reviews.",
    "total_reviews": len(df),
    "average_rating": df['reviews.rating'].mean(),
    "average_sentiment": df['sentiment_polarity'].mean(),
    "topics": json.loads(topics) if topics else {},
    "visualizations": ["rating_distribution.png", "sentiment_over_time.png", "word_cloud.png", "helpfulness_vs_rating.png", "topic_distribution.png"]
}
print(json.dumps(report, indent=4))

## Provide your publication link below:

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import traceback
from openai import AzureOpenAI

# ==========================================
# 1. AZURE OPENAI SETUP
# ==========================================
# These will be provided in your class. 
# For testing, ensure they are in your .env file or environment variables.
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "your-endpoint-here")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "your-api-key-here")
API_VERSION = "2024-02-15-preview"
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4") # Update to your deployment name

# Initialize Azure Client
client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=API_KEY,
    api_version=API_VERSION
)

# ==========================================
# 2. CUSTOM TOOLS (Function Calling)
# ==========================================
# Tool 1: Summarize Dataset
def summarize_dataset(file_path: str) -> str:
    """Reads the dataset and returns a statistical summary and sample reviews."""
    try:
        df = pd.read_csv(file_path)
        total_rows = len(df)
        avg_rating = df['reviews.rating'].mean() if 'reviews.rating' in df.columns else "N/A"
        
        # Sample top 5 and bottom 5 reviews for context
        if 'reviews.rating' in df.columns and 'reviews.text' in df.columns:
            top_reviews = df[df['reviews.rating'] >= 4]['reviews.text'].head(3).tolist()
            bad_reviews = df[df['reviews.rating'] <= 2]['reviews.text'].head(3).tolist()
        else:
            top_reviews, bad_reviews = [], []

        summary = {
            "total_reviews": total_rows,
            "average_rating": avg_rating,
            "sample_positive_reviews": top_reviews,
            "sample_critical_reviews": bad_reviews
        }
        return json.dumps(summary)
    except Exception as e:
        return json.dumps({"error": str(e)})

# Tool 2: Generate Visualization
def generate_rating_distribution_plot(file_path: str) -> str:
    """Generates a seaborn plot of the ratings distribution and saves it to disk."""
    try:
        df = pd.read_csv(file_path)
        if 'reviews.rating' not in df.columns:
            return json.dumps({"error": "Column 'reviews.rating' not found in dataset."})
        
        plt.figure(figsize=(8, 5))
        sns.countplot(data=df, x='reviews.rating', palette='viridis')
        plt.title('Distribution of Customer Ratings')
        plt.xlabel('Rating')
        plt.ylabel('Count')
        
        plot_path = "ratings_distribution.png"
        plt.savefig(plot_path)
        plt.close()
        
        return json.dumps({"status": "success", "message": f"Plot saved to {plot_path}"})
    except Exception as e:
        return json.dumps({"error": str(e)})

# Map tool names to functions for the execution loop
AVAILABLE_TOOLS = {
    "summarize_dataset": summarize_dataset,
    "generate_rating_distribution_plot": generate_rating_distribution_plot
}

# OpenAI Tool Schemas
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "summarize_dataset",
            "description": "Get a statistical summary and sample texts from the reviews dataset.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "Path to the CSV file"}
                },
                "required": ["file_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_rating_distribution_plot",
            "description": "Generates a matplotlib/seaborn visualization of the ratings distribution.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "Path to the CSV file"}
                },
                "required": ["file_path"]
            }
        }
    }
]

# ==========================================
# 3. BASE AGENT CLASS
# ==========================================
class Agent:
    def __init__(self, name: str, role: str, system_prompt: str, tools: list = None):
        self.name = name
        self.role = role
        self.system_prompt = system_prompt
        self.tools = tools or []
        self.max_iterations = 5 # Safety Guard 1

    def run(self, user_message: str) -> str:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message}
        ]
        
        iterations = 0
        while iterations < self.max_iterations:
            iterations += 1
            try:
                # Call Azure OpenAI
                response = client.chat.completions.create(
                    model=DEPLOYMENT_NAME,
                    messages=messages,
                    tools=self.tools if self.tools else None,
                    tool_choice="auto" if self.tools else None,
                    temperature=0.2
                )
                
                response_message = response.choices[0].message
                
                # Check for tool calls
                if response_message.tool_calls:
                    messages.append(response_message) # Append the assistant's tool call request
                    
                    for tool_call in response_message.tool_calls:
                        function_name = tool_call.function.name
                        function_args = json.loads(tool_call.function.arguments)
                        
                        # Execute the tool
                        function_to_call = AVAILABLE_TOOLS.get(function_name)
                        if function_to_call:
                            function_response = function_to_call(**function_args)
                            
                            # Append the tool response
                            messages.append({
                                "tool_call_id": tool_call.id,
                                "role": "tool",
                                "name": function_name,
                                "content": function_response,
                            })
                    # Loop back to let the LLM read the tool response
                    continue 
                else:
                    # Final text response
                    return response_message.content

            except Exception as e:
                # Safety Guard 2: Graceful error handling
                error_msg = f"Agent '{self.name}' encountered an error: {str(e)}\n{traceback.format_exc()}"
                print(error_msg)
                return f"Error executing {self.name}. Please check the logs and your API keys."
        
        return f"Agent '{self.name}' reached maximum iterations without completing the task."

# ==========================================
# 4. MULTI-AGENT ORCHESTRATION (Sequential Pipeline)
# ==========================================
def run_pipeline(csv_file_path: str):
    # Agent 1: Data Loader & Visualizer
    data_agent = Agent(
        name="DataLoader",
        role="Data Analyst",
        system_prompt="You are an expert Data Analyst. Your job is to use tools to extract statistical summaries and generate visual plots from CSV files. Summarize the basic facts of the data clearly.",
        tools=tools_schema
    )
    
    # Agent 2: Sentiment & Topic Modeler
    sentiment_agent = Agent(
        name="SentimentModeller",
        role="NLP Specialist",
        system_prompt="You are an NLP Specialist. Analyze raw data summaries and sample reviews. Identify the main themes, common complaints, and primary reasons for positive reviews. Output a structured analysis.",
        tools=None # Consumes text, doesn't need external tools
    )
    
    # Agent 3: Business Insight Strategist
    business_agent = Agent(
        name="BusinessInsightAgent",
        role="Product Manager",
        system_prompt="You are a brilliant E-commerce Product Manager. Take the sentiment analysis provided by the NLP specialist and translate it into actionable business recommendations. Format your output as a final Markdown report.",
        tools=None
    )

    # Execution (Sequential Pipeline)
    st.write("🕵️‍♂️ **Agent 1 (Data Analyst)** is analyzing the dataset and generating visualizations...")
    data_output = data_agent.run(f"Please summarize the dataset at '{csv_file_path}' and generate a rating distribution plot.")
    with st.expander("View Data Analyst Output"):
        st.write(data_output)

    st.write("🧠 **Agent 2 (NLP Specialist)** is extracting themes and sentiments...")
    sentiment_output = sentiment_agent.run(f"Here is the data summary:\n{data_output}\n\nPlease analyze the sentiment and extract key topics from the sample reviews.")
    with st.expander("View NLP Specialist Output"):
        st.write(sentiment_output)

    st.write("💼 **Agent 3 (Business Strategist)** is drafting the final report...")
    final_report = business_agent.run(f"Based on this NLP analysis, write a final actionable business report:\n{sentiment_output}")
    
    return final_report

# ==========================================
# 5. STREAMLIT UI (The Bonus +10)
# ==========================================
def main():
    st.set_page_config(page_title="E-commerce Review Intelligence Agents", layout="wide")
    st.title("🤖 Multi-Agent E-commerce Review Intelligence")
    st.markdown("This system uses three specialized AI Agents to analyze Amazon product reviews, generate visualizations, and provide actionable business insights.")

    # File Uploader
    uploaded_file = st.file_uploader("Upload your Amazon Reviews CSV (min 500 rows)", type=["csv"])

    if uploaded_file is not None:
        # Save uploaded file temporarily for pandas to read
        temp_path = "temp_dataset.csv"
        with open(temp_path, "wb") as f:
            f.write(uploaded_file.getbuffer())
        
        # Display preview
        df_preview = pd.read_csv(temp_path)
        st.write(f"Dataset Loaded Successfully! Rows: {len(df_preview)}")
        st.dataframe(df_preview.head(3))
        
        if st.button("🚀 Run Multi-Agent Analysis"):
            with st.spinner("Orchestrating AI Agents Pipeline..."):
                final_report = run_pipeline(temp_path)
                
                st.success("Analysis Complete!")
                st.markdown("---")
                st.subheader("📊 Generated Visualization")
                if os.path.exists("ratings_distribution.png"):
                    st.image("ratings_distribution.png", caption="Ratings Distribution by Agent Tool")
                
                st.markdown("---")
                st.subheader("📑 Final Business Report (Structured Output)")
                st.markdown(final_report)

if __name__ == "__main__":
    main()